In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
from matplotlib.cm import ScalarMappable
import lightgbm as lgb
import shap
from shap.plots import beeswarm as shap_beeswarm
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import mean_absolute_error
import itertools
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.family']        = 'serif'
plt.rcParams['font.serif']         = ['Times New Roman']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['axes.linewidth']     = 2
plt.rcParams['xtick.major.width']  = 2
plt.rcParams['ytick.major.width']  = 2

# ══════════════════════════════════════════════════════════
# 1. 超参数设置
# ══════════════════════════════════════════════════════════
DATA_PATH      = 'B0005_特征数据集.xlsx'
TRAIN_RATIO    = 0.6
NULL_N_RUNS    = 50
NULL_THRESHOLD = 0.75
RANDOM_SEED    = 42
OUTPUT_PREFIX  = 'B0005'
SEARCH_SPACE = {
    'num_leaves':        list(range(50, 305, 5)),    
    'feature_fraction':  [round(v, 2) for v in
                          np.arange(0.5, 1.01, 0.1)], 
    'bagging_fraction':  [round(v, 2) for v in
                          np.arange(0.8, 1.01, 0.1)],
    'bagging_freq':      list(range(0, 9)),            
}
LGB_FIXED = {
    'objective':          'regression',
    'metric':             'mae',           
    'boosting_type':      'gbdt',
    'learning_rate':      0.05,
    'min_child_samples':  5,              
    'verbosity':          -1,
    'random_state':       RANDOM_SEED,
}
LGB_ROUNDS       = 300
N_CV_FOLDS       = 5      
N_RANDOM_SEARCH  = 50     
CMAP = mcolors.LinearSegmentedColormap.from_list(
    "custom_cmap", ["blue", "#4B0082", "red"]
)

# ══════════════════════════════════════════════════════════
# 2. 加载数据
# ══════════════════════════════════════════════════════════
df = pd.read_excel(DATA_PATH)
target_col = df.columns[-1]
selected_features = [
    'ACC', 'TECD', 'TEVR', 'TETR', 'CDET', 'VDET',
    'TEVD', 'TRET', 'ADV', 'SDV', 'CCCT', 'CCDT', 
    'HCTT', 'HDT', 'HDTT', 'ICP', 'ICPL', 'ICA', 
    'IC_max', 'DTVFPA', 'DTCA', 'DTC_quar3', 'DTC_quar1', 'DTC_quar2'
]
X = df[selected_features]
y = df[target_col]
print(f"特征数: {len(selected_features)}  |  样本数: {len(df)}")

# ══════════════════════════════════════════════════════════
# 3. 划分训练/测试集
# ══════════════════════════════════════════════════════════
split_idx = int(len(df) * TRAIN_RATIO)
X_train = X.iloc[:split_idx].reset_index(drop=True)
X_test  = X.iloc[split_idx:].reset_index(drop=True)
y_train = y.iloc[:split_idx].reset_index(drop=True)
y_test  = y.iloc[split_idx:].reset_index(drop=True)
print(f"训练集: {len(X_train)}  |  测试集: {len(X_test)}")

# ══════════════════════════════════════════════════════════
# 4. LightGBM 超参数搜索
# ══════════════════════════════════════════════════════════
print("\n" + "="*55)
print("  LightGBM 超参数搜索（Random Grid Search）")
print("="*55)
np.random.seed(RANDOM_SEED)
all_keys   = list(SEARCH_SPACE.keys())
all_values = list(SEARCH_SPACE.values())
total_combinations = 1
for v in all_values:
    total_combinations *= len(v)
print(f"搜索空间总组合数: {total_combinations}")
print(f"随机采样候选数:   {N_RANDOM_SEARCH}")
sampled_idx = np.random.choice(total_combinations,
                                size=min(N_RANDOM_SEARCH, total_combinations),
                                replace=False)
candidate_params = []
for idx in sampled_idx:
    combo = {}
    tmp = idx
    for k, vals in zip(reversed(all_keys), reversed(all_values)):
        combo[k] = vals[tmp % len(vals)]
        tmp //= len(vals)
    candidate_params.append(combo)

def time_series_cv_mae(params, X_tr, y_tr, n_splits=N_CV_FOLDS):
    n      = len(X_tr)
    fold_size = n // (n_splits + 1)
    maes   = []
    full_params = {**LGB_FIXED, **params}

    for fold in range(1, n_splits + 1):
        train_end = fold * fold_size
        val_end   = min(train_end + fold_size, n)
        if val_end <= train_end:
            continue
        X_cv_train = X_tr.iloc[:train_end]
        y_cv_train = y_tr.iloc[:train_end]
        X_cv_val   = X_tr.iloc[train_end:val_end]
        y_cv_val   = y_tr.iloc[train_end:val_end]

        dtrain = lgb.Dataset(X_cv_train, label=y_cv_train)
        model  = lgb.train(full_params, dtrain,
                           num_boost_round=LGB_ROUNDS,
                           callbacks=[lgb.log_evaluation(period=-1)])
        preds  = model.predict(X_cv_val)
        maes.append(mean_absolute_error(y_cv_val, preds))

    return np.mean(maes) if maes else float('inf')

best_mae    = float('inf')
best_params = candidate_params[0]
search_log  = []

print(f"\n开始搜索（共 {len(candidate_params)} 组）...")
for i, params in enumerate(candidate_params):
    mae = time_series_cv_mae(params, X_train, y_train)
    search_log.append({**params, 'CV_MAE': mae})
    if mae < best_mae:
        best_mae    = mae
        best_params = params
    if (i + 1) % 10 == 0:
        print(f"  进度: {i+1}/{len(candidate_params)}  "
              f"当前最优 MAE={best_mae:.6f}", end='\r')

print(f"\n\n✅ 超参数搜索完成")
print(f"最优参数: {best_params}")
print(f"最优 CV-MAE: {best_mae:.6f}")
search_df = pd.DataFrame(search_log).sort_values('CV_MAE').reset_index(drop=True)
print(f"\nTop5 参数组合:")
print(search_df.head(5).to_string(index=False))

# ══════════════════════════════════════════════════════════
# 5. 最优参数构建最终 LightGBM
# ══════════════════════════════════════════════════════════
BEST_LGB_PARAMS = {**LGB_FIXED, **best_params}

print(f"\n最终使用的完整参数:")
for k, v in BEST_LGB_PARAMS.items():
    print(f"  {k}: {v}")

def train_lgb(X_tr, y_tr, params=None):
    p      = params if params else BEST_LGB_PARAMS
    dtrain = lgb.Dataset(X_tr, label=y_tr)
    model  = lgb.train(p, dtrain, num_boost_round=LGB_ROUNDS,
                       callbacks=[lgb.log_evaluation(period=-1)])
    return model

# ══════════════════════════════════════════════════════════
# 6. Null Importance 分析
# ══════════════════════════════════════════════════════════
print("\n" + "="*55)
print("  Null Importance 分析（最优参数 LightGBM）")
print("="*55)

model_real = train_lgb(X_train, y_train)
real_split = dict(zip(selected_features,
                       model_real.feature_importance('split')))
real_gain  = dict(zip(selected_features,
                       model_real.feature_importance('gain')))

null_split = {f: [] for f in selected_features}
null_gain  = {f: [] for f in selected_features}

print(f"计算Null重要性（{NULL_N_RUNS}次打乱）...")
for i in range(NULL_N_RUNS):
    y_shuffled = y_train.sample(frac=1.0, random_state=i).values
    m = train_lgb(X_train, y_shuffled)
    for j, f in enumerate(selected_features):
        null_split[f].append(m.feature_importance('split')[j])
        null_gain[f].append(m.feature_importance('gain')[j])
    if (i + 1) % 10 == 0:
        print(f"  进度: {i+1}/{NULL_N_RUNS}", end='\r')
print()

null_rows = []
for f in selected_features:
    ns = np.array(null_split[f])
    ng = np.array(null_gain[f])
    rs, rg = real_split[f], real_gain[f]
    null_rows.append({
        'Feature':         f,
        'Real_Split':      rs,
        'Real_Gain':       rg,
        'Null_Split_Mean': ns.mean(),
        'Null_Gain_Mean':  ng.mean(),
        'Split_Ratio':     rs / (ns.mean() + 1e-8),
        'Gain_Ratio':      rg / (ng.mean() + 1e-8),
        'Split_pvalue':    (ns >= rs).mean(),
        'Gain_pvalue':     (ng >= rg).mean(),
        'Pass': (rs > np.percentile(ns, NULL_THRESHOLD * 100)) and
                (rg > np.percentile(ng, NULL_THRESHOLD * 100))
    })

null_df = pd.DataFrame(null_rows)\
            .sort_values('Real_Gain', ascending=False)\
            .reset_index(drop=True)
final_features    = null_df[null_df['Pass']]['Feature'].tolist()
rejected_features = null_df[~null_df['Pass']]['Feature'].tolist()

print("\nNull Importance 结果:")
print(null_df[['Feature','Real_Split','Real_Gain',
               'Split_Ratio','Gain_Ratio',
               'Split_pvalue','Gain_pvalue','Pass']].to_string(index=False))
print(f"\n通过验证: {len(final_features)} 个 → {final_features}")
print(f"未通过:   {len(rejected_features)} 个 → {rejected_features}")

# ══════════════════════════════════════════════════════════
# 7. 可视化一：Null Importance 直方图
# ══════════════════════════════════════════════════════════
feat_order = null_df['Feature'].tolist()
n_feat     = len(feat_order)

fig_null, axes_null = plt.subplots(
    n_feat, 2,
    figsize=(14, 2.8 * n_feat),
    facecolor='white'
)
fig_null.suptitle('Null Importance Analysis',
                   fontsize=18, fontweight='bold', y=1.002)

for i, feat in enumerate(feat_order):
    ax_s = axes_null[i, 0]
    ax_g = axes_null[i, 1]
    ns, ng = null_split[feat], null_gain[feat]
    rs, rg = real_split[feat], real_gain[feat]
    passed = null_df[null_df['Feature'] == feat]['Pass'].values[0]
    vcolor = '#1565C0' if passed else '#B71C1C'

    n_s, _, _ = ax_s.hist(ns, bins=20, alpha=0.7,
                           color='skyblue', label='Disrupted Label')
    ax_s.axvline(rs, color=vcolor, linestyle='--',
                 linewidth=3, label='Correct Label')
    ax_s.set_xlabel('Importance Split', fontsize=12)
    ax_s.set_ylabel(f'{feat}\nFrequency', fontsize=11)
    ax_s.set_ylim(0, max(n_s) * 1.3)
    ax_s.legend(prop={'family': 'Times New Roman', 'size': 10})
    ax_s.grid(True, alpha=0.3)
    for sp in ax_s.spines.values():
        sp.set_linewidth(1.5)

    n_g, _, _ = ax_g.hist(ng, bins=20, alpha=0.7,
                           color='lightgreen', label='Disrupted Label')
    ax_g.axvline(rg, color=vcolor, linestyle='--',
                 linewidth=3, label='Correct Label')
    ax_g.set_xlabel('Importance Gain', fontsize=12)
    ax_g.set_ylabel('Frequency', fontsize=11)
    ax_g.set_ylim(0, max(n_g) * 1.3)
    ax_g.legend(prop={'family': 'Times New Roman', 'size': 10})
    ax_g.grid(True, alpha=0.3)
    ax_g.set_title('✔ Pass' if passed else '✘ Reject',
                   color=vcolor, fontsize=11, fontweight='bold')
    for sp in ax_g.spines.values():
        sp.set_linewidth(1.5)

plt.tight_layout()
plt.savefig(f'{OUTPUT_PREFIX}_null_importance.png',
            dpi=180, bbox_inches='tight')
plt.close()
print(f"\n✅ Null Importance图已保存")

# ══════════════════════════════════════════════════════════
# 8. SHAP 分析
# ══════════════════════════════════════════════════════════
use_features = final_features if len(final_features) >= 3 else selected_features
print(f"\nSHAP分析使用特征: {use_features}")

X_shap_train = X_train[use_features]
X_shap_all   = X[use_features]

model_shap  = train_lgb(X_shap_train, y_train)
explainer   = shap.TreeExplainer(model_shap)
shap_values = explainer(X_shap_all)

mean_abs_shap   = np.abs(shap_values.values).mean(axis=0)
shap_series     = pd.Series(mean_abs_shap, index=use_features)\
                    .sort_values(ascending=False)
sorted_features = shap_series.index.tolist()
sorted_shap_vals = shap_series.values

color_norm = mcolors.Normalize(vmin=sorted_shap_vals.min(),
                                vmax=sorted_shap_vals.max())
bar_colors = CMAP(color_norm(sorted_shap_vals))

print("\nSHAP特征重要性排序:")
for f, v in zip(sorted_features, sorted_shap_vals):
    print(f"  {f}: {v:.6f}")

# ───────────────────────── 玫瑰图数据 ───────────────────────────────
num_vars           = len(sorted_features)
percentages        = sorted_shap_vals / sorted_shap_vals.sum() * 100
widths             = np.full(num_vars, 2 * np.pi / num_vars)
thetas             = np.linspace(0, 2 * np.pi, num_vars, endpoint=False)
inner_ring_base    = 1.5
inner_ring_width   = 0.3
colored_ring_width = sorted_shap_vals / sorted_shap_vals.max() * 4.0
inner_heights = np.array([inner_ring_base + inner_ring_width * (i % 2)
                           for i in range(num_vars)])
total_lengths = inner_heights + colored_ring_width
inner_colors  = ['#D0D0D0' if i % 2 == 0 else '#F0F0F0'
                 for i in range(num_vars)]

# ══════════════════════════════════════════════════════════
# 9. 可视化二：玫瑰图(a) + 蜂巢图(b)
# ══════════════════════════════════════════════════════════
fig_h = max(7, num_vars * 1.4)
fig_shap = plt.figure(figsize=(26, fig_h), facecolor='white')

left_margin    = 0.05;  right_margin  = 0.04
bottom_margin  = 0.09;  top_margin    = 0.07
space_between  = 0.01;  colorbar_width = 0.01
plot_bottom = bottom_margin
plot_height = 1 - bottom_margin - top_margin
total_w     = 1 - left_margin - right_margin - space_between
left_plot_w = total_w * 0.56;  right_plot_w = total_w * 0.40

# 颜色条
cbar_left = left_margin
ax_cbar   = fig_shap.add_axes([cbar_left, plot_bottom,
                                colorbar_width, plot_height])
sm   = ScalarMappable(cmap=CMAP, norm=color_norm)
cbar = fig_shap.colorbar(sm, cax=ax_cbar, orientation='vertical')
cbar.set_label('', size=18, labelpad=5)
cbar.set_ticks([])
cbar.ax.yaxis.set_ticks_position('left')
ax_cbar.text(0.5,  1.01, 'High', transform=ax_cbar.transAxes,
             ha='center', va='bottom', fontsize=28)
ax_cbar.text(0.5, -0.01, 'Low',  transform=ax_cbar.transAxes,
             ha='center', va='top', fontsize=28)
cbar.outline.set_visible(False)
ax_cbar.text(-1.8, 0.5, 'Mean |SHAP Value|',
             transform=ax_cbar.transAxes,
             fontsize=28, rotation=90, va='center')

# 主条形图
main_ax_left = cbar_left + colorbar_width + 0.02
ax_bar = fig_shap.add_axes([main_ax_left, plot_bottom,
                             left_plot_w, plot_height])
ax_bar.xaxis.tick_bottom()
ax_bar.xaxis.set_label_position("bottom")
ax_bar.invert_xaxis()
ax_bar.barh(range(num_vars), sorted_shap_vals,
            color=bar_colors, height=0.45)
ax_bar.invert_yaxis()
ax_bar.set_xlabel('Mean |SHAP Value|', size=28, labelpad=20)
ax_bar.set_yticks([])
ax_bar.spines[['left', 'top']].set_visible(False)
ax_bar.spines['right'].set_position(('data', 0))
ax_bar.spines['right'].set_visible(True)
ax_bar.spines['bottom'].set_visible(True)
ax_bar.tick_params(axis='x', which='major',
                   direction='in', labelsize=26, length=6, pad=8)
ax_bar.xaxis.set_minor_locator(ticker.AutoMinorLocator(5))
ax_bar.tick_params(axis='x', which='minor', direction='in', length=4)
label_pad = sorted_shap_vals.max() * 0.01
for i, feature in enumerate(sorted_features):
    ax_bar.text(label_pad, i, feature,
                ha='right', va='center', color='black', fontsize=26)
ax_bar.text(-0.02, 1.00, '(a)', transform=ax_bar.transAxes,
            fontsize=32, weight='bold', ha='left', va='top')

# 内嵌玫瑰图
inset_size   = min(left_plot_w, plot_height) * 1.5
center_x     = main_ax_left - 0.12 + (min(left_plot_w, plot_height) * 0.72) / 2
center_y     = plot_bottom + 0.02   + (min(left_plot_w, plot_height) * 0.72) / 2 + 0.05
inset_left   = center_x - inset_size / 2
inset_bottom = center_y - inset_size / 2
ax_radial = fig_shap.add_axes(
    [inset_left, inset_bottom, inset_size, inset_size],
    projection='polar'
)
ax_radial.patch.set_alpha(0)
ax_radial.bar(thetas, inner_heights, width=widths,
              color=inner_colors, align='edge',
              edgecolor='white', linewidth=1.5)
ax_radial.bar(thetas, colored_ring_width, width=widths,
              bottom=inner_heights, color=bar_colors,
              align='edge', edgecolor='white', linewidth=1.5)
for i in range(num_vars):
    angle  = thetas[i] + widths[i] / 2
    radius = total_lengths[i] + 0.4
    ax_radial.text(angle, radius, f'{percentages[i]:.1f}%',
                   ha='center', va='center', fontsize=20)
ax_radial.set_yticklabels([])
ax_radial.set_xticklabels([])
ax_radial.spines['polar'].set_visible(False)
ax_radial.grid(False)
ax_radial.set_theta_zero_location('N')
ax_radial.set_theta_direction(-1)
ax_radial.set_ylim(0, max(total_lengths) + 1.5)

# 蜂巢图
right_left = main_ax_left + left_plot_w + space_between
ax_bee = fig_shap.add_axes([right_left, plot_bottom,
                             right_plot_w, plot_height])
shap_beeswarm(shap_values, max_display=len(sorted_features),
              ax=ax_bee, show=False, color=CMAP, plot_size=None)
ax_bee.set_yticklabels([])
ax_bee.set_ylabel('')
ax_bee.set_xlabel('SHAP Value (impact on model output)', fontsize=28)
ax_bee.tick_params(axis='x', labelsize=26)
ax_bee.text(0.05, 1.00, '(b)', transform=ax_bee.transAxes,
            fontsize=32, fontweight='bold', va='top', ha='right')
if len(fig_shap.axes) > 3:
    cbar_bee = fig_shap.axes[-1]
    cbar_bee.set_ylabel('Feature Value', size=26, rotation=270, labelpad=16)
    cbar_bee.tick_params(labelsize=24)

plt.savefig(f'{OUTPUT_PREFIX}_SHAP_rose_beeswarm.png',
            dpi=180, bbox_inches='tight', facecolor='white')
plt.close()
print(f"✅ SHAP组合图已保存")

# ══════════════════════════════════════════════════════════
# 10. 超参搜索可视化
# ══════════════════════════════════════════════════════════
fig_search, ax_s = plt.subplots(figsize=(12, 5), facecolor='white')
ax_s.plot(search_df['CV_MAE'].values, color='#1565C0',
          linewidth=1.5, marker='o', markersize=3, alpha=0.7)
ax_s.axhline(best_mae, color='#E53935', linestyle='--',
             linewidth=1.5, label=f'Best MAE = {best_mae:.6f}')
ax_s.set_xlabel('Candidate Rank', fontsize=13)
ax_s.set_ylabel('CV-MAE', fontsize=13)
ax_s.set_title('LightGBM Hyperparameter Search Results\n'
               f'(metric=mae, min_child_samples=5, '
               f'num_leaves∈[50,300], feature_fraction∈[0.5,1.0])',
               fontsize=12)
ax_s.legend(fontsize=11)
ax_s.spines[['top', 'right']].set_visible(False)
ax_s.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig(f'{OUTPUT_PREFIX}_hyperparam_search.png',
            dpi=180, bbox_inches='tight', facecolor='white')
plt.close()
print(f"✅ 超参搜索图已保存")

# ══════════════════════════════════════════════════════════
# 10. 导出结果到 Excel
# ══════════════════════════════════════════════════════════
with pd.ExcelWriter(f'{OUTPUT_PREFIX}_NullImportance_SHAP结果.xlsx',
                    engine='openpyxl') as writer:

    null_df.to_excel(writer, sheet_name='Null_Importance', index=False)

    pd.DataFrame({
        'Feature':       sorted_features,
        'Mean_AbsSHAP':  sorted_shap_vals,
        'Contribution%': percentages
    }).to_excel(writer, sheet_name='SHAP_排名', index=False)

    search_df.head(20).to_excel(writer, sheet_name='超参搜索Top20', index=False)

    pd.DataFrame([{
        '参数': k, '最优值': v
    } for k, v in {**BEST_LGB_PARAMS,
                   'Best_CV_MAE': round(best_mae, 6),
                   'LGB_ROUNDS': LGB_ROUNDS}.items()])\
      .to_excel(writer, sheet_name='最优LightGBM参数', index=False)

    pd.DataFrame({
        '项目': ['总样本', '训练集', '测试集', 'Null迭代次数',
                  'Null阈值分位数', '超参搜索候选数',
                  '通过验证特征数', 'SHAP使用特征数'],
        '值':   [len(df), len(X_train), len(X_test), NULL_N_RUNS,
                  f'P{int(NULL_THRESHOLD*100)}', len(candidate_params),
                  len(final_features), len(use_features)]
    }).to_excel(writer, sheet_name='实验设置', index=False)

print(f"✅ 结果已保存: {OUTPUT_PREFIX}_NullImportance_SHAP结果.xlsx")